[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Declarative Models &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: the college's classes, `deans_list`, and the staff hierarchy of `Person`, `Instructor`
and `Advisor`. Run it first. The tasks do not depend on one another, and the last cell removes the
scratch folder.


In [1]:
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (JSON, CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table,
                        UniqueConstraint, create_engine, event, func, insert, inspect, select, text)
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy.pool import StaticPool
from sqlalchemy.schema import CreateTable

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}

def show_ddl(table, engine):
    """Print the CREATE TABLE statement a Table becomes on an engine's database."""
    for line in str(CreateTable(table).compile(engine)).strip().splitlines():
        print("   ", line.rstrip().replace("\t", "    "))

engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    @property
    def level(self):
        """100 for an introductory course, 200 for the next, read from the number in the code."""
        return int(self.code.split("-")[1]) // 100 * 100

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def deans_list(session, term, minimum=3.0):
    """The students whose grade point average for a term is at least `minimum`, best first, then by name."""
    graded = (
        select(Student, Enrollment, Course)
        .join(Enrollment)
        .join(Section)
        .join(Course)
        .join(Term)
        .where(Term.name == term, Enrollment.grade.is_not(None))
    )
    points, credits = {}, {}
    for student, enrollment, course in session.execute(graded):
        points[student] = points.get(student, 0) + enrollment.grade_points * course.credits
        credits[student] = credits.get(student, 0) + course.credits
    averages = {student: round(points[student] / credits[student], 2) for student in points}
    chosen = [(average, student) for student, average in averages.items() if average >= minimum]
    return sorted(chosen, key=lambda pair: (-pair[0], pair[1].name))


class StaffBase(DeclarativeBase):
    pass


class Person(StaffBase):
    __tablename__ = "people"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    kind: Mapped[str] = mapped_column(String(20))

    __mapper_args__ = {"polymorphic_on": "kind", "polymorphic_identity": "person"}

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Instructor(Person):
    department: Mapped[str | None] = mapped_column(String(50))
    __mapper_args__ = {"polymorphic_identity": "instructor"}


class Advisor(Person):
    office: Mapped[str | None] = mapped_column(String(20))
    __mapper_args__ = {"polymorphic_identity": "advisor"}


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


**1.** A `Room` class, and its table.


In [2]:
class RoomBase(DeclarativeBase):
    pass


class Room(RoomBase):
    __tablename__ = "rooms"

    id: Mapped[int] = mapped_column(primary_key=True)
    building: Mapped[str] = mapped_column(String(50))
    number: Mapped[str] = mapped_column(String(10))
    seats: Mapped[int | None]


show_ddl(Room.__table__, engine)


    CREATE TABLE rooms (
        id INTEGER NOT NULL,
        building VARCHAR(50) NOT NULL,
        number VARCHAR(10) NOT NULL,
        seats INTEGER,
        PRIMARY KEY (id)
    )


`seats` is the one column without `NOT NULL`, because its annotation allows `None`. The number is a
string, since room numbers such as `B-204` are not numbers.


**2.** Objects, then rows.


In [3]:
with Session(engine) as session:
    objects = session.scalars(select(Course).where(Course.credits == 4).order_by(Course.code)).all()
    rows = session.execute(select(Course.code, Course.title).where(Course.credits == 4).order_by(Course.code)).all()

print(type(objects[0]).__name__, objects)
print(type(rows[0]).__name__, rows)


Course [Course('BIO-101', 4), Course('CHE-110', 4), Course('MAT-120', 4), Course('MAT-121', 4)]
Row [('BIO-101', 'Introduction to Biology'), ('CHE-110', 'General Chemistry'), ('MAT-120', 'Calculus I'), ('MAT-121', 'Calculus II')]


**3.** Grade points, from a property.


In [4]:
with Session(engine) as session:
    fall = select(Enrollment).where(Enrollment.student_id == 3, Enrollment.section_id.between(21, 30))
    for enrollment in session.scalars(fall.order_by(Enrollment.section_id)):
        print(enrollment, "| worth", enrollment.grade_points)


Enrollment(student 3, section 22, 'C+') | worth 2.3
Enrollment(student 3, section 26, 'F') | worth 0.0
Enrollment(student 3, section 29, 'B+') | worth 3.3


The F is worth 0.0 points, not `None`: `grade_points` returns `None` only when there is no grade at
all.


**4.** A count for every program.


In [5]:
with Session(engine) as session:
    counts = session.execute(select(Student.program, func.count()).group_by(Student.program).order_by(Student.program)).all()

print(counts)


[('Biology', 5), ('Computer Science', 5), ('History', 5), ('Mathematics', 5), ('Psychology', 5)]


Rows, because the statement asked for a column and a count, not for the class. No single student
stands behind a count of five, so there is no object to return, and the result is the same one Core
would give.


**5.** A kind with no columns of its own.


In [6]:
class Librarian(Person):
    __mapper_args__ = {"polymorphic_identity": "librarian"}


library = college_engine()
StaffBase.metadata.create_all(library)
with Session(library) as session:
    session.add_all([Person(name="Ada Registrar"), Instructor(name="Dr. Okafor", department="Biology"),
                     Advisor(name="Ms. Lin", office="B-204"), Librarian(name="Mr. Berg")])
    session.commit()
    print(session.scalars(select(Person).order_by(Person.id)).all())
library.dispose()


[Person('Ada Registrar'), Instructor('Dr. Okafor'), Advisor('Ms. Lin'), Librarian('Mr. Berg')]


A kind of person needs only a class and an identity: its rows sit in `people` like the others, with
`kind` set to `librarian`, and `select(Person)` turns each row into the class its `kind` names.


**6.** The dean's list for Fall 2024.


In [7]:
with Session(engine) as session:
    for average, student in deans_list(session, "Fall 2024", minimum=2.8):
        print(f"{average:.2f}  {student}")


3.01  Student('Grace Lin', 'Computer Science')
2.90  Student('Pavel Novak', 'Biology')


Only the nine students who started in Fall 2024 have grades for it, so the list draws on fewer
students than the later terms do.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Declarative Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/07-declarative-models.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
